
# Scrape notable CJEU rulings from Wikipedia

This notebook scrapes:

> https://en.wikipedia.org/wiki/List_of_European_Court_of_Justice_rulings

It extracts the **case name**, **case number**, and the Wikipedia **topic/subtopic** in which the case appears.

### Normalisation

- Historical bare case numbers such as `26/62` or `12/85` are normalised to `C-26/62` and `C-12/85`.
- Existing `C-...` case numbers are kept as `C-...`.
- General Court cases such as `T-17/10` are preserved as `T-17/10`.
- Appeal suffixes such as `C-501/06 P` are preserved.
- Simple joined-case ranges such as `C-222/05 to C-225/05` are expanded to individual case numbers.
- The output is **one row per case number**, which makes it easy to merge against an existing case/proceeding dataset.

The notebook also retains the original bullet text (`raw_text`) so that questionable rows can be checked manually.


In [1]:

# If needed, uncomment this:
# %pip install requests beautifulsoup4 pandas lxml

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, NavigableString

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)

URL = "https://en.wikipedia.org/wiki/List_of_European_Court_of_Justice_rulings"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; academic-research-notebook/1.0; "
        "contact: local-user)"
    )
}


In [2]:

# Download the article

response = requests.get(URL, headers=HEADERS, timeout=30)
response.raise_for_status()

print("HTTP status:", response.status_code)
print("Downloaded characters:", len(response.text))

soup = BeautifulSoup(response.text, "html.parser")

content = (
    soup.select_one("div.mw-content-ltr.mw-parser-output")
    or soup.select_one("div.mw-parser-output")
)

if content is None:
    raise RuntimeError(
        "Could not locate the Wikipedia article body. "
        "The page structure may have changed."
    )


HTTP status: 200
Downloaded characters: 673045


In [3]:

# Helpers for cleaning text and extracting case numbers

DASHES = "\u2010\u2011\u2012\u2013\u2014\u2212"
DASH_TRANSLATION = str.maketrans({c: "-" for c in DASHES})

EXPLICIT_CASE_RE = re.compile(
    r"\b([CTF])\s*[-‐‑‒–—−]?\s*(\d{1,4})/(\d{2,4})(?:\s+([Pp]))?\b"
)

BARE_CASE_RE = re.compile(
    r"(?<![\w/-])(\d{1,4})/(\d{2,4})(?![\w/-])"
)

JOINED_SHORTHAND_RE = re.compile(
    r"\bjoined\s+cases?\s+(\d{1,4})\s+(?:and|&)\s+(\d{1,4})/(\d{2,4})\b",
    flags=re.I,
)

RANGE_RE = re.compile(
    r"\b([CTF])\s*[-‐‑‒–—−]?\s*(\d{1,4})/(\d{2,4})"
    r"\s+(?:to|through|-)\s+"
    r"(?:([CTF])\s*[-‐‑‒–—−]?\s*)?(\d{1,4})/(\d{2,4})\b",
    flags=re.I,
)


def clean_text(text):
    text = str(text).translate(DASH_TRANSLATION)
    text = re.sub(r"\[\s*\d+\s*\]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \n\t")


def normalise_case_number(prefix, number, year, suffix=None):
    prefix = prefix.upper()
    result = f"{prefix}-{int(number)}/{year}"
    if suffix:
        result += f" {suffix.upper()}"
    return result


def extract_case_numbers(text):
    text = clean_text(text)
    found = []

    # 1. Explicit C-/T-/F- references
    for m in EXPLICIT_CASE_RE.finditer(text):
        found.append(
            normalise_case_number(m.group(1), m.group(2), m.group(3), m.group(4))
        )

    # 2. Expand simple explicit ranges
    for m in RANGE_RE.finditer(text):
        p1, n1, y1, p2, n2, y2 = m.groups()
        p2 = p2 or p1

        if p1.upper() == p2.upper() and y1 == y2:
            start, end = int(n1), int(n2)
            if start <= end and (end - start) <= 50:
                for n in range(start, end + 1):
                    found.append(normalise_case_number(p1, n, y1))

    # 3. Old shorthand: "joined cases 90 and 91/63"
    shorthand_spans = []
    for m in JOINED_SHORTHAND_RE.finditer(text):
        n1, n2, year = m.groups()
        found.extend([
            normalise_case_number("C", n1, year),
            normalise_case_number("C", n2, year),
        ])
        shorthand_spans.append(m.span())

    # 4. Bare historical references: 26/62 -> C-26/62
    explicit_spans = [m.span() for m in EXPLICIT_CASE_RE.finditer(text)]

    for m in BARE_CASE_RE.finditer(text):
        start, end = m.span()

        if any(a <= start and end <= b for a, b in explicit_spans):
            continue
        if any(a <= start and end <= b for a, b in shorthand_spans):
            continue

        before = text[max(0, start - 25):start]
        after = text[end:end + 8]

        # Opinion 2/94 is not C-2/94
        if re.search(r"\bOpinion\s*$", before, flags=re.I):
            continue

        # Avoid legislation such as 96/62/EC
        if re.match(r"\s*/\s*(?:EC|EU|EEC|ECSC)\b", after, flags=re.I):
            continue

        found.append(normalise_case_number("C", m.group(1), m.group(2)))

    return list(dict.fromkeys(found))


tests = [
    "Example 12/85 [1986] ECR 123",
    "Case C-23/23: Example v Commission",
    "Simba Toys T-450/09 [2014]",
    "Joined Cases C-430/93 and C-431/93 Van Schijndel",
    "Joined cases 90 and 91/63",
    "Joined Cases C-222/05 to C-225/05",
    "Opinion 2/94 [1996] ECR I-1759",
]

for x in tests:
    print(x, "->", extract_case_numbers(x))


Example 12/85 [1986] ECR 123 -> ['C-12/85']
Case C-23/23: Example v Commission -> ['C-23/23']
Simba Toys T-450/09 [2014] -> ['T-450/09']
Joined Cases C-430/93 and C-431/93 Van Schijndel -> ['C-430/93', 'C-431/93']
Joined cases 90 and 91/63 -> ['C-90/63', 'C-91/63']
Joined Cases C-222/05 to C-225/05 -> ['C-222/05', 'C-225/05', 'C-223/05', 'C-224/05']
Opinion 2/94 [1996] ECR I-1759 -> []


In [4]:

# Helpers for article structure and case-name extraction

def list_item_text(li):
    parts = []

    for child in li.children:
        if getattr(child, "name", None) in {"ul", "ol"}:
            continue

        if isinstance(child, NavigableString):
            parts.append(str(child))
        else:
            parts.append(child.get_text(" ", strip=True))

    return clean_text(" ".join(parts))


def heading_text(tag):
    text = tag.get_text(" ", strip=True)
    text = re.sub(r"\[\s*edit\s*\]", "", text, flags=re.I)
    return clean_text(text)


def extract_case_name(text):
    text = clean_text(text)
    text = re.sub(r"\s*\^\s*\d+\s*", " ", text)

    # Name before ", joined cases ..."
    m = re.search(r",?\s+joined\s+cases?\b", text, flags=re.I)
    if m and m.start() > 2:
        candidate = text[:m.start()].strip(" ,;:-")
        if len(candidate) >= 3:
            return candidate

    spans = []

    for m in EXPLICIT_CASE_RE.finditer(text):
        spans.append(m.span())

    for m in BARE_CASE_RE.finditer(text):
        before = text[max(0, m.start() - 15):m.start()]

        if re.search(r"\bOpinion\s*$", before, flags=re.I):
            continue

        if any(a <= m.start() and m.end() <= b for a, b in spans):
            continue

        spans.append(m.span())

    spans = sorted(spans)

    if not spans:
        return None

    first_start = spans[0][0]
    last_end = spans[-1][1]

    prefix = text[:first_start].strip(" ,;:-")
    prefix_clean = re.sub(
        r"^(?:case|cases|joined case|joined cases)\s*$",
        "",
        prefix,
        flags=re.I,
    ).strip()

    if len(prefix_clean) >= 3:
        prefix_clean = re.sub(
            r",?\s+(?:joined\s+)?cases?$",
            "",
            prefix_clean,
            flags=re.I,
        ).strip(" ,;:-")

        if prefix_clean:
            return prefix_clean

    tail = text[last_end:].strip(" ,;:-")

    tail = re.sub(
        r"^(?:(?:and|to)\s+(?:[CTF]\s*-?\s*)?\d{1,4}/\d{2,4}\s*)+",
        "",
        tail,
        flags=re.I,
    ).strip(" ,;:-")

    tail = re.split(r"\s+-\s+|:\s+(?=[A-Z])", tail, maxsplit=1)[0]
    tail = re.split(r"\.\s+(?=[A-Z])", tail, maxsplit=1)[0]
    tail = re.split(r"\s+\[\d{4}\]|\s+ECR\b", tail, maxsplit=1, flags=re.I)[0]

    return tail.strip(" ,;:-.") or None


In [5]:

# Scrape all relevant list items while retaining topic headings

rows = []

current_h2 = None
current_h3 = None
current_h4 = None

for tag in content.find_all(["h2", "h3", "h4", "li"]):

    if tag.name == "h2":
        current_h2 = heading_text(tag)
        current_h3 = None
        current_h4 = None
        continue

    if tag.name == "h3":
        current_h3 = heading_text(tag)
        current_h4 = None
        continue

    if tag.name == "h4":
        current_h4 = heading_text(tag)
        continue

    raw_text = list_item_text(tag)
    case_numbers = extract_case_numbers(raw_text)

    if not case_numbers:
        continue

    case_name = extract_case_name(raw_text)

    for case_number in case_numbers:
        rows.append({
            "case_name": case_name,
            "case_number": case_number,
            "topic": current_h2,
            "subtopic": current_h3,
            "subsubtopic": current_h4,
            "raw_text": raw_text,
        })

df = pd.DataFrame(rows)

df = df.drop_duplicates(
    subset=["case_name", "case_number", "topic", "subtopic", "subsubtopic"]
).reset_index(drop=True)

print(f"Extracted {len(df):,} case-number rows.")
print(f"Unique case numbers: {df['case_number'].nunique():,}")
print(f"Unique parsed names: {df['case_name'].nunique(dropna=True):,}")

df.head(30)


Extracted 339 case-number rows.
Unique case numbers: 260
Unique parsed names: 274


,case_name,case_number,topic,subtopic,subsubtopic,raw_text
0,Primacy ( ECJ,C-6/64,None,None,None,Primacy ( ECJ Case 6/64 )
1,Van Gend en Loos v. Nederlandse Administratie der Belastingen,C-26/62,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Van Gend en Loos v. Nederlandse Administratie der Belastingen 26/62 ECR 1
2,Franz Grad,C-9/70,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Franz Grad 9/70 ECR-825
3,Commission v Italy,C-39/72,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Commission v Italy 39/72 ECR 101
4,Reyners,C-2/74,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Reyners 2/74 ECR 631
5,Amsterdam Bulb,C-50/76,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Amsterdam Bulb 50/76 ECR 137
6,Zaera,C-126/86,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Zaera 126/86 ECR 3697
7,Azienda Agricola,C-403/98,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Azienda Agricola C-403/98 ECR I-103
8,Steinberg,T-17/10,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Steinberg T-17/10 625
9,Sharif University,T-181/13,Principles of Union Law,Direct effect,"Treaties, Regulations and Decisions",Sharif University T-181/13 607


In [6]:

# Basic quality checks

print("Case-number prefix counts:")
display(
    df["case_number"]
    .str.extract(r"^([A-Z]+)-", expand=False)
    .value_counts(dropna=False)
)

print("\nRows where the case name could not be parsed:")
display(df[df["case_name"].isna()].head(30))

print("\nPotentially suspicious very short names:")
display(
    df[
        df["case_name"].fillna("").str.len().between(1, 4)
    ][["case_name", "case_number", "raw_text"]].head(30)
)


Case-number prefix counts:


case_number
C    330
T      9
Name: count, dtype: int64


Rows where the case name could not be parsed:


,case_name,case_number,topic,subtopic,subsubtopic,raw_text



Potentially suspicious very short names:


,case_name,case_number,raw_text
37,Nold,C-4/73,"Nold 4/73 ECR 491, §13"
45,Beus,C-5/67,Beus 5/67 ECR 83
70,In,C-92/11,"In case C-92/11, RWE Vertrieb AG v Verbraucherzentrale Nordrhein-Westfalen [ de ] , the court ruled that the use of standard terms in a consumer gas supply contract which refle..."
208,),C-7/13,"Case C-7/13 Skandia America Corp (USA) v. Skatteverket (the Swedish Tax Agency ), 14 September 2014, affected certain member states who operate a form of ""establishment-only"" V..."
209,),C-210/04,"Case C-7/13 Skandia America Corp (USA) v. Skatteverket (the Swedish Tax Agency ), 14 September 2014, affected certain member states who operate a form of ""establishment-only"" V..."
318,↑,C-94/12,"↑ Case C-94/12, Swm Costruzioni 2 SpA and Mannocchi Luigino DI v Provincia di Fermo , InfoCuria , paragraph 39, accessed 24 July 2023"
324,↑,C-149/77,"↑ Case C-149/77, Gabrielle Defrenne v Société Anonyme Belge de Navigation Aérienne Sabena , 15 June 1978, accessed 28 January 2021"
325,↑,C-549/13,"↑ Case C-549/13 Bundesdruckerei GmbH v. Stadt Dortmund , 18 September 2014, accessed 28 January 2021"
326,↑,C-22/08,"↑ Case C-22/08 and C-23/08, Vatsouras and Koupatantze v ARGE , 12 March 2009, accessed 5 December 2024"
327,↑,C-23/08,"↑ Case C-22/08 and C-23/08, Vatsouras and Koupatantze v ARGE , 12 March 2009, accessed 5 December 2024"


In [7]:

# Compact dataset for merging against your own CJEU data

cases = (
    df[["case_name", "case_number", "topic", "subtopic", "subsubtopic"]]
    .drop_duplicates()
    .sort_values(["case_number", "case_name"], na_position="last")
    .reset_index(drop=True)
)

cases.head(50)


,case_name,case_number,topic,subtopic,subsubtopic
0,Fratelli Costanzo,C-103/88,Principles of Union Law,Direct effect,Directives
1,Amministrazione delle Finanze dello Stato v Simmenthal SpA (1978),C-106/77,Fifty-seven pre-accession cases,None,None
2,Simmenthal II,C-106/77,Principles of Union Law,Primacy,None
3,Marleasing,C-106/89,Principles of Union Law,Primacy,None
4,Commission v Italian Republic,C-107/92,Procurement,Public procurement,None
5,"↑ EUR-Lex, Judgment of the Court, 2 August 1993, in",C-107/92,Notes,None,None
6,"(conjoined), Larentia + Minerva and Marenave , 16 July 2015",C-108/14,Taxation,VAT groups,None
7,"RI.SAN. v Comune di Ischia , Italia Lavoro SpA, formerly GEPI SpA and Ischia Ambiente SpA : the treaty principles relating to freedom of movement, freedom of establishment and ...",C-108/98,Internal market,Non-application of single market principles,None
8,"↑ EUR-Lex, RI.SAN. Srl and Comune di Ischia, Italia Lavoro SpA, formerly GEPI SpA, Ischia Ambiente SpA",C-108/98,Notes,None,None
9,"(conjoined), Larentia + Minerva and Marenave , 16 July 2015",C-109/14,Taxation,VAT groups,None


In [10]:

# Optional: Court of Justice cases only
# This excludes T-/F- cases while keeping historical bare cases normalised to C-...

cjeu_only = cases[cases["case_number"].str.startswith("C-")].copy()

print("All extracted rows:", len(cases))
print("C- cases only:", len(cjeu_only))

cjeu_only.head(10)
cjeu_only.tail(10)


All extracted rows: 339
C- cases only: 330


,case_name,case_number,topic,subtopic,subsubtopic
320,Faccini Dori,C-91/92,Principles of Union Law,Direct effect,Directives
321,Faccini Dori,C-91/92,State liability,None,None
322,Paola Faccini Dori v Recreb Srl (1994),C-91/92,Fifty-seven pre-accession cases,None,None
323,In,C-92/11,Consumer protection,None,None
324,Zuckerfabrik Süderdithmarschen AG v Hauptzollamt Itzehoe and Zuckerfabrik Soest GmbH v Hauptzollamt Paderborn (1991),C-92/89,Fifty-seven pre-accession cases,None,None
325,Swm Contruzioni 2 SpA and Mannochi Luigino DI v,C-94/12,Procurement,Public procurement,None
326,↑,C-94/12,Notes,None,None
327,"Lennartz v Finanzamt München III , published 11 July 1991, accessed 26 March 2021",C-97/90,Notes,None,None
328,Lennartz v Finanzamt München III: reference for a preliminary ruling on VAT paid on the purchase of capital goods,C-97/90,Taxation,Value added tax,None
329,"Commission v Belgium : contracting authorities have ""a degree of choice"" as to which procurement procedure they follow for each procurement exercise, but ""once they have issued...",C-97/94,Procurement,Public procurement,None


In [9]:

# Export

df.to_csv("wikipedia_cjeu_notable_cases_full.csv", index=False)
cases.to_csv("wikipedia_cjeu_notable_cases.csv", index=False)
cjeu_only.to_csv("wikipedia_cjeu_notable_cases_C_only.csv", index=False)

print("Saved:")
print("  wikipedia_cjeu_notable_cases_full.csv")
print("  wikipedia_cjeu_notable_cases.csv")
print("  wikipedia_cjeu_notable_cases_C_only.csv")


Saved:
  wikipedia_cjeu_notable_cases_full.csv
  wikipedia_cjeu_notable_cases.csv
  wikipedia_cjeu_notable_cases_C_only.csv



## Notes

This Wikipedia article is a curated list of **notable** rulings, not a complete register of CJEU proceedings. The `raw_text` column is intentionally retained because some bullets use unusual prose rather than a uniform citation format.

For research use, it makes most sense to treat this as an **importance / notable-case indicator** that can later be merged into your main court dataset by `case_number`, rather than as a source for the complete universe of cases.
